# Using the `qbraid-SDK`
qBraid SDK is our flagship Python toolkit to run platform agnostic quantum programs. We have built the `qbraid.runtime` module to interface with multiple quantum software packages and their respective vendors. 

In [ ]:
%%capture

%pip install 'qbraid[qiskit,cirq,braket,visualization]'

In [ ]:
import qbraid

qbraid.__version__

## Using the `QbraidProvider` and Devices
- We use the provider abstraction to associate together all the devices available via qBraid
- Each provider has a list of supported devices which can be retrieved via the `.get_devices()` method

In [ ]:
from qbraid.runtime import QbraidProvider

provider = QbraidProvider()

In [ ]:
provider.get_devices()

### Choose AWS Statevector Simulator

In [ ]:
aws_sim = provider.get_device("aws:aws:sim:sv1")
aws_sim

In [ ]:
aws_sim._target_spec

## Define the quantum circuit

We will build the following circuit representing [the **Bell** state](https://en.wikipedia.org/wiki/Bell_state) - 

$$ | \Psi \rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}} $$

## Running Simulator : AWS Statevector Simulator

###  Submit Cirq circuit to AWS simulator

#### Build a Cirq circuit
Cirq is Google's quantum computing library

In [ ]:
import cirq

# Create 2 qubits
qubits = [cirq.LineQubit(i) for i in range(2)]

# Define the circuit
bell_circuit = cirq.Circuit()

# Apply a Hadamard gate to the first qubit
bell_circuit.append(cirq.H(qubits[0]))

# Apply CNOT gates to entangle the qubits
bell_circuit.append(cirq.CNOT(qubits[0], qubits[1]))

# Measure all qubits
bell_circuit.append(cirq.measure(*qubits, key="result"))

In [ ]:
print(bell_circuit)

#### Execute cirq on AWS simulator

In [ ]:
job = aws_sim.run(bell_circuit, shots=1000)
print(job.result().data.measurement_counts)

###  Submit qiskit circuit to AWS simulator 

#### Build a Qiskit circuit
Qiskit is IBM's quantum programming library

In [ ]:
from qiskit import QuantumCircuit

# Create a 2-qubit quantum circuit with 2 classical bits for measurement
qiskit_qc = QuantumCircuit(2)

qiskit_qc.h(0)
qiskit_qc.cx(0, 1)

qiskit_qc.measure_all()

In [ ]:
print(qiskit_qc)

#### Execute qiskit on AWS simulator

In [ ]:
job = aws_sim.run(qiskit_qc, shots=1000)
print(job.result().data.measurement_counts)

## Running on real hardware : Rigetti Cepheus

In [ ]:
rigetti_device = provider.get_device("aws:rigetti:qpu:cepheus-1-108q")
rigetti_device

In [ ]:
rigetti_device.metadata()

We will again utilise the **Qiskit circuit** that we developed in the above block -

In [ ]:
num_shots = 50  # Review device pricing to calculate the cost of the experiment. Do not increase to a very high value.
job = rigetti_device.run(qiskit_qc, shots=num_shots)
print(job)

In [ ]:
job.status()

In [ ]:
print("Final result: ", job.result())

In [ ]:
print("Final counts : ", job.result().data.measurement_counts)

- We can clearly see that the majority counts are from the $|00\rangle$ and the $|11\rangle$ state
- Since this is a **real quantum device**, we are bound to get some **noisy results**. This is highlighted from the fact that 
we get some shots from other possible states i.e. $|01\rangle$ and $|10\rangle$ 

## Running on real hardware : IQM Garnet

In [ ]:
iqm_garnet_device = provider.get_device("aws:iqm:qpu:garnet")
iqm_garnet_device

In [ ]:
iqm_garnet_device.metadata()

In [ ]:
num_shots = 50  # Review device pricing to calculate the cost of the experiment. Do not increase to a very high value.
iqm_job = iqm_garnet_device.run(qiskit_qc, shots=num_shots)
print(iqm_job)

In [ ]:
iqm_job.status()

In [ ]:
print("Final result: ", iqm_job.result())

In [ ]:
print("Final counts : ", iqm_job.result().data.measurement_counts)

- We can again see that the majority counts are from the $|00\rangle$ and the $|11\rangle$ state
- Since this is a **real quantum device**, we are bound to get some **noisy results**. This is highlighted from the fact that 
we get some shots from other possible states i.e. $|01\rangle$ and $|10\rangle$ 

<div class="alert alert-block alert-info">
<b>Copyright Notice:</b> 
    All rights reserved © [2026] qBraid. This notebook is part of the qBraid-Lab-Demo repository.
The qBraid-Lab-Demo is licensed under the Apache License, Version 2.0.
You may obtain a copy of the License at <https://www.apache.org/licenses/LICENSE-2.0>.
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
</div>